# 05 -- P-Median Allocation Model (Phase 3) and Scenario/Evaluation Framework (Phase 4)

Implements outline sections:

- **3.4.3 Phase 3: P-Median Location-Allocation Model** -- the capacitated MILP (O3)
- **3.4.4 Phase 4: Scenario Analysis and Evaluation Framework** -- the 12-configuration core
  grid, K/Uj/k sensitivity, optional service-radius scenario, and M1-M4 (O4)

**Model type:** Mixed-Integer Linear Program (MILP), solved with PuLP/CBC. $x_j$ is always
integer; $y_{ij}$ is continuous $[0,1]$ by current default (a `binary_y` switch is kept so this
can be flipped to $\{0,1\}$ once the Camden-scale comparison test settles the question); $s_j$
is continuous and unconstrained above.

**Slack is NOT in the objective** (current decision): $s_j$ only appears in constraint C1,
uncapped and unpenalised, purely to keep the model always feasible. Because $s_j$ can absorb
any unmet demand for any $K$, **the model is provably feasible for every tested $K$** -- there
is no need for the feasibility-safety "K-bump" logic the earlier version of this notebook used;
it has been removed here as no longer necessary.

Reads `demand_london.csv` from `04_demand_estimation.ipynb` directly -- `e_i` is already a
column there, so no separate supply file needs to be reloaded.

## 0. Setup and load data

In [3]:
import pandas as pd
import numpy as np
import geopandas as gpd
from scipy.spatial import cKDTree
import pulp
import os
import time

BASE = "/Users/alexia/Documents/CASA/Dissertation"
OUTPUT_DIR = os.path.join(BASE, "05_processed")
figures_dir = os.path.join(BASE, "06_outputs/figures")
os.makedirs(figures_dir, exist_ok=True)

demand = pd.read_csv(os.path.join(OUTPUT_DIR, "demand_london.csv"))
print("demand_london.csv:", demand.shape, "| columns:", list(demand.columns))
assert len(demand) == 4_994, f"Expected 4,994 London LSOAs, got {len(demand)}"

lsoa_boundaries = gpd.read_file(os.path.join(BASE, "03_data/demand/spatial/LSOA_2021_EW_BGC_V5.shp"))
if lsoa_boundaries.crs is None or lsoa_boundaries.crs.to_epsg() != 27700:
    lsoa_boundaries = lsoa_boundaries.to_crs(epsg=27700)

london_codes = set(demand["lsoa_code"])
lsoa_geo = lsoa_boundaries[lsoa_boundaries["LSOA21CD"].isin(london_codes)].copy()
lsoa_geo = lsoa_geo.rename(columns={"LSOA21CD": "lsoa_code"})[["lsoa_code", "geometry"]]
lsoa_geo["centroid"] = lsoa_geo.geometry.centroid
lsoa_geo["cx"] = lsoa_geo["centroid"].x
lsoa_geo["cy"] = lsoa_geo["centroid"].y

master = demand.merge(lsoa_geo[["lsoa_code", "cx", "cy"]], on="lsoa_code", how="left").reset_index(drop=True)
assert master["cx"].isna().sum() == 0, "Some LSOAs failed to match a boundary centroid."

n = len(master)
coords = master[["cx", "cy"]].to_numpy()
print(f"Master table: {n} LSOAs, I = J = all Greater London LSOA centroids.")

demand_london.csv: (4994, 14) | columns: ['lsoa_code', 'lsoa_name', 'Hi', 'Ci', 'HiCi', 'e_i', 'ubar_i', 'has_evse_i', 'Ui_multiplier', 'IMD_i', 'D_A', 'D_B', 'D_C', 'D_D']
Master table: 4994 LSOAs, I = J = all Greater London LSOA centroids.


## 1. London-specific deprivation deciles

`imd_london_clean.csv`'s `income_decile` is the official England-wide **national** decile
(confirmed while building 04 -- it comes straight from IoD2025's national ranking, not a
London-only recalculation). Outline 3.4.4 requires London-specific deciles for M3/M4, so they
are recomputed here directly from `IMD_i`, ranked within the 4,994 Greater London LSOAs only.

Decile 1 = most deprived 10% of *London* LSOAs (highest $IMD_i$); Decile 10 = least deprived,
matching the national convention's direction so "decile 1 = most deprived" means the same thing
throughout.

In [4]:
# 用method="first"打散所有并列值，保证rank严格唯一，qcut就能切出尽可能均匀的10组
income_rank = master["IMD_i"].rank(method="first", ascending=False)  # rank 1 = 最高IMD_i = 最贫困
master["income_decile_london"] = pd.qcut(income_rank, 10, labels=False) + 1

decile_check = master.groupby("income_decile_london")["IMD_i"].agg(["count", "min", "max"])
print("London-specific deciles (1 = most deprived, 10 = least deprived):")
print(decile_check)
assert master["income_decile_london"].between(1, 10).all()

decile_msg = "Decile groups should each hold roughly 499-500 LSOAs."
assert decile_check["count"].between(490, 510).all(), decile_msg

London-specific deciles (1 = most deprived, 10 = least deprived):
                      count    min    max
income_decile_london                     
1                       500  0.466  0.998
2                       499  0.396  0.466
3                       499  0.346  0.396
4                       500  0.304  0.346
5                       499  0.262  0.304
6                       499  0.219  0.262
7                       500  0.173  0.218
8                       499  0.129  0.173
9                       499  0.084  0.129
10                      500  0.010  0.084


## 2. Candidate assignment sets (k-NN pruning) and K0 calibration

$k=40$ nearest LSOA centroids per demand node (self always included) -- a computational
sparsification, not a policy distance rule; full $I \times J$ would need ~25 million $y_{ij}$
variables, intractable for CBC at this scale.

$$K_0 = \frac{\sum_i D_i^{(0)}}{\sum_j e_j + p_{ref}}, \quad p_{ref} = 5{,}000$$

Fixed independently of whichever $p$ is actually being tested, and shared across all 12 core
configurations so differences between scenarios are not created by silently recalibrating $K$.

In [5]:
K_DEFAULT = 40  # k-NN candidate pruning, baseline value

def build_candidate_sets(coords, k):
    n_local = len(coords)
    tree = cKDTree(coords)
    _, nbr = tree.query(coords, k=min(k, n_local))
    cand = [set(np.atleast_1d(row).tolist()) for row in nbr]
    for i in range(n_local):
        cand[i].add(i)
    return cand

cand_default = build_candidate_sets(coords, K_DEFAULT)
avg_candidates = np.mean([len(c) for c in cand_default])
print(f"k={K_DEFAULT}: average candidate set size per demand node = {avg_candidates:.1f}")

P_REF = 5_000
sum_D0 = master["D_A"].sum()   # D_i^(0) -- alpha=0 baseline recognised demand
sum_ej = master["e_i"].sum()
K0 = sum_D0 / (sum_ej + P_REF)

print(f"Sigma D_i^(0) (Scenario A total): {sum_D0:,.2f}")
print(f"Sigma e_j (existing on-street locations): {sum_ej:,.0f}")
print(f"K0 = Sigma D_i^(0) / (Sigma e_j + {P_REF:,}) = {K0:.6f}")

k=40: average candidate set size per demand node = 40.0
Sigma D_i^(0) (Scenario A total): 347,150.71
Sigma e_j (existing on-street locations): 21,364
K0 = Sigma D_i^(0) / (Sigma e_j + 5,000) = 13.167604


## 3. Model function

Returns per-LSOA mean assignment distance $\bar d_i = \sum_j d_{ij} y_{ij}$ directly (not the
full $y_{ij}$ matrix, to keep memory bounded across 12+ full-London solves) plus solver
diagnostics and the self-assignment share ($y_{ii}$, needed for the pre-evaluation checks).

In [9]:
def solve_p_median(demand_col, p, K, master_df, coords, cand,
                    Uj=150, binary_y=False, service_radius=None,
                    time_limit=300, frac_gap=0.02, msg=False,
                    slack_penalty=None):
    """
    Capacitated p-median MILP (outline 3.4.3, C1-C4).
    C1 (elastic, PENALISED slack): sum_i Di*yij <= K*(ej+xj) + sj
    C2: sum_j yij = 1 for each i
    C3: sum_j xj = p
    C4: xj integer in [0,Uj]; yij in [0,1] (or {0,1} if binary_y=True)
    Optional C5: yij = 0 if dij > service_radius (metres)

    Slack is now penalised in the objective (version B). With I=J and d_ii=0, an unpenalised
    slack (version A) makes yii=1 for almost every LSOA a zero-cost, always-feasible solution --
    the solver has no reason not to take it, regardless of K/Uj/k. Penalising slack removes that
    free lunch: leaving demand unserved is only ever chosen when it is genuinely unavoidable
    (total London-wide capacity cannot absorb total demand), not as a shortcut.
    """
    Di = master_df[demand_col].to_numpy(dtype=float)
    ej = master_df["e_i"].to_numpy(dtype=float)
    n_local = len(master_df)

    def d(i, j):
        return float(np.hypot(coords[i, 0] - coords[j, 0], coords[i, 1] - coords[j, 1]))

    if service_radius is not None:
        cand_use = [{j for j in cand[i] if d(i, j) <= service_radius or j == i} for i in range(n_local)]
    else:
        cand_use = cand

    # M must exceed the longest possible single reassignment distance in London, so that leaving
    # 1 unit of demand in slack is always strictly worse than moving it anywhere, however far.
    # 100x gives a comfortable margin -- slack is used only when truly unavoidable.
    if slack_penalty is None:
        max_dist = float(np.hypot(np.ptp(coords[:, 0]), np.ptp(coords[:, 1])))
        slack_penalty = 100.0 * max_dist

    prob = pulp.LpProblem("p_median", pulp.LpMinimize)
    x = pulp.LpVariable.dicts("x", range(n_local), lowBound=0, upBound=Uj, cat="Integer")

    if binary_y:
        y = {(i, j): pulp.LpVariable(f"y_{i}_{j}", cat="Binary")
             for i in range(n_local) for j in cand_use[i]}
    else:
        y = {(i, j): pulp.LpVariable(f"y_{i}_{j}", lowBound=0, upBound=1)
             for i in range(n_local) for j in cand_use[i]}

    s = pulp.LpVariable.dicts("s", range(n_local), lowBound=0)

    prob += (pulp.lpSum(Di[i] * d(i, j) * y[(i, j)] for (i, j) in y)
             + slack_penalty * pulp.lpSum(s[j] for j in range(n_local)))   # distance cost + penalised slack

    for i in range(n_local):
        prob += pulp.lpSum(y[(i, j)] for j in cand_use[i]) == 1                  # C2

    served_by = {j: [] for j in range(n_local)}
    for (i, j) in y:
        served_by[j].append(i)
    for j in range(n_local):
        if served_by[j]:
            prob += (pulp.lpSum(Di[i] * y[(i, j)] for i in served_by[j])
                     <= K * (ej[j] + x[j]) + s[j])                               # C1

    prob += pulp.lpSum(x[j] for j in range(n_local)) == p                        # C3

    t0 = time.time()
    solver_result = prob.solve(pulp.PULP_CBC_CMD(msg=int(msg), timeLimit=time_limit, gapRel=frac_gap))
    solve_seconds = time.time() - t0

    xj = np.array([int(round(x[j].value() or 0)) for j in range(n_local)])
    sj = np.array([float(s[j].value() or 0) for j in range(n_local)])

    dbar_i = np.zeros(n_local)
    y_ii = np.zeros(n_local)
    for (i, j), var in y.items():
        val = var.value() or 0.0
        if val:
            dbar_i[i] += d(i, j) * val
            if i == j:
                y_ii[i] = val

    return {
        "xj": xj, "sj": sj, "dbar_i": dbar_i, "y_ii": y_ii,
        "status": pulp.LpStatus[solver_result],
        "solve_seconds": solve_seconds,
        "n_assignment_links": len(y),
        "K_used": K,
        "slack_penalty_used": slack_penalty,
        "objective": float(pulp.value(prob.objective)),
    }

print("solve_p_median defined (slack now penalised in the objective).")

solve_p_median defined (slack now penalised in the objective).


## 4. M1-M4 evaluation function

All four metrics use the fixed baseline weight $D_i^{(0)}$ (Scenario A, $\alpha=0$) regardless
of which scenario is being evaluated, so metric changes reflect the allocation outcome, not a
shifting evaluation ruler.

In [10]:
def weighted_percentile(values, weights, percentile):
    order = np.argsort(values)
    v_sorted = values[order]
    w_sorted = weights[order]
    cum_w = np.cumsum(w_sorted)
    cutoff = percentile / 100.0 * cum_w[-1]
    idx = int(np.searchsorted(cum_w, cutoff))
    idx = min(idx, len(v_sorted) - 1)
    return v_sorted[idx]


def evaluate_m1_m4(dbar_i, dbar_i_baseline_same_p, D0, xj, p, decile_london):
    # M1: efficiency cost relative to alpha=0 at the same p
    numerator = np.sum(D0 * (dbar_i - dbar_i_baseline_same_p))
    denominator = np.sum(D0 * dbar_i_baseline_same_p)
    M1 = 100.0 * numerator / denominator if denominator > 0 else np.nan

    # M2: D0-weighted 90th percentile of LSOA-level mean assignment distance
    M2 = weighted_percentile(dbar_i, D0, 90)

    # M3: deprivation accessibility gap (London-specific deciles, D0-weighted group means)
    most_mask = decile_london == 1
    least_mask = decile_london == 10
    dbar_most = np.average(dbar_i[most_mask], weights=D0[most_mask])
    dbar_least = np.average(dbar_i[least_mask], weights=D0[least_mask])
    M3 = dbar_most - dbar_least

    # M4: equity allocation ratio (most deprived decile's capacity share / its baseline demand share)
    cap_share_most = xj[most_mask].sum() / p if p > 0 else np.nan
    demand_share_most = D0[most_mask].sum() / D0.sum()
    M4 = cap_share_most / demand_share_most if demand_share_most > 0 else np.nan

    return {"M1_efficiency_cost_pct": M1, "M2_tail_accessibility_m": M2,
            "M3_deprivation_gap_m": M3, "M4_equity_allocation_ratio": M4,
            "dbar_baseline_weighted_m": float(np.average(dbar_i, weights=D0))}

print("evaluate_m1_m4 defined.")

evaluate_m1_m4 defined.


## 5. Baseline pre-evaluation checks

Solved once at the reference configuration ($\alpha=0$, $p=5{,}000$) before running the full
12-config grid. Confirms the allocation is a meaningful result, not a degenerate solution
($y_{ii}\approx1$ everywhere would mean every LSOA is just serving itself, with $\bar d_i$
collapsing to ~0 and M1's denominator vanishing).

In [11]:
D0 = master["D_A"].to_numpy()  # D_i^(0), fixed evaluation weight throughout

print("Solving reference baseline (alpha=0, p=5,000) for pre-evaluation checks...")
baseline_ref = solve_p_median("D_A", 5_000, K0, master, coords, cand_default, Uj=150, binary_y=False)

mean_dist_baseline = float(np.average(baseline_ref["dbar_i"], weights=D0))
mean_self_assign_share = float(np.mean(baseline_ref["y_ii"]))

print(f"Status: {baseline_ref['status']}, solve time: {baseline_ref['solve_seconds']:.1f}s")
print(f"D0-weighted mean assignment distance: {mean_dist_baseline:,.1f} m")
print(f"Mean self-assignment share (y_ii, averaged across LSOAs): {mean_self_assign_share:.4f}")

checks_passed = {
    "Baseline mean distance > 0": mean_dist_baseline > 0,
    "Not dominated by y_ii approx 1 (mean y_ii < 0.9)": mean_self_assign_share < 0.9,
    "Solver did not report Infeasible (should be mathematically impossible with elastic slack)":
        baseline_ref["status"] != "Infeasible",
}
for label, passed in checks_passed.items():
    print(f"{'PASS' if passed else 'FAIL'} -- {label}")

if mean_dist_baseline <= 0 or mean_self_assign_share >= 0.9:
    raise AssertionError(
        "Baseline allocation looks degenerate (near-zero distances / near-total self-assignment). "
        "Do not proceed to the core grid or M1-M4 until this is investigated -- check K0, Uj, "
        "and candidate pruning before re-running."
    )
print("\nBaseline looks like a meaningful (non-degenerate) allocation -- proceeding.")

Solving reference baseline (alpha=0, p=5,000) for pre-evaluation checks...
Status: Optimal, solve time: 31.1s
D0-weighted mean assignment distance: 290.6 m
Mean self-assignment share (y_ii, averaged across LSOAs): 0.7525
PASS -- Baseline mean distance > 0
PASS -- Not dominated by y_ii approx 1 (mean y_ii < 0.9)
PASS -- Solver did not report Infeasible (should be mathematically impossible with elastic slack)

Baseline looks like a meaningful (non-degenerate) allocation -- proceeding.


## 6. Core scenario grid: 12 configurations ($\alpha \times p$)

$\alpha \in \{0, 0.1, 0.3, 0.5\}$ (Scenarios A/B/C/D) x $p \in \{2{,}000, 5{,}000, 8{,}000\}$.
Same $K_0$, $U_j=150$, $k=40$ throughout -- only $\alpha$ and $p$ vary across these 12 runs.
This is the slowest cell in the notebook (up to 12 full-London MILP solves); progress is
printed after each one.

In [12]:
SCENARIOS = {"A": 0.0, "B": 0.1, "C": 0.3, "D": 0.5}
P_VALUES = [2_000, 5_000, 8_000]

core_results = {}   # (scenario_label, p) -> result dict
run_log = []

for p in P_VALUES:
    for label in SCENARIOS:
        demand_col = f"D_{label}"
        t0 = time.time()
        result = solve_p_median(demand_col, p, K0, master, coords, cand_default, Uj=150, binary_y=False)
        core_results[(label, p)] = result
        run_log.append({
            "scenario": label, "alpha": SCENARIOS[label], "p": p,
            "status": result["status"], "solve_seconds": result["solve_seconds"],
            "objective": result["objective"], "slack_total": float(result["sj"].sum()),
        })
        print(f"Scenario {label} (alpha={SCENARIOS[label]}), p={p}: "
              f"status={result['status']}, {result['solve_seconds']:.1f}s, "
              f"total slack={result['sj'].sum():,.1f}")

run_log_df = pd.DataFrame(run_log)
print("\n=== Solver run log ===")
print(run_log_df.to_string(index=False))

assert (run_log_df["status"] != "Infeasible").all(), (
    "At least one core configuration reported Infeasible -- this should be mathematically "
    "impossible with elastic, unpenalised slack in C1. Investigate before trusting any M1-M4 "
    "results computed from this run."
)

Scenario A (alpha=0.0), p=2000: status=Optimal, 11.8s, total slack=150,434.0
Scenario B (alpha=0.1), p=2000: status=Optimal, 15.4s, total slack=156,503.5
Scenario C (alpha=0.3), p=2000: status=Optimal, 13.3s, total slack=168,681.6
Scenario D (alpha=0.5), p=2000: status=Optimal, 15.0s, total slack=180,878.6
Scenario A (alpha=0.0), p=5000: status=Optimal, 29.9s, total slack=110,931.2
Scenario B (alpha=0.1), p=5000: status=Optimal, 64.9s, total slack=117,000.7
Scenario C (alpha=0.3), p=5000: status=Optimal, 38.8s, total slack=129,178.8
Scenario D (alpha=0.5), p=5000: status=Optimal, 23.2s, total slack=141,375.8
Scenario A (alpha=0.0), p=8000: status=Optimal, 93.4s, total slack=71,428.4
Scenario B (alpha=0.1), p=8000: status=Optimal, 64.1s, total slack=77,497.9
Scenario C (alpha=0.3), p=8000: status=Optimal, 74.9s, total slack=89,676.0
Scenario D (alpha=0.5), p=8000: status=Optimal, 55.4s, total slack=101,873.0

=== Solver run log ===
scenario  alpha    p  status  solve_seconds    objectiv

## 7. M1-M4 across all 12 configurations

Each scenario is compared to its own $\alpha=0$ baseline **at the same $p$** (M1's own-$p$
baseline requirement), while $D_i^{(0)}$ (Scenario A demand) is used as the fixed evaluation
weight everywhere.

In [13]:
decile_london = master["income_decile_london"].to_numpy()

m1_m4_rows = []
for p in P_VALUES:
    dbar_baseline_same_p = core_results[("A", p)]["dbar_i"]
    for label in SCENARIOS:
        result = core_results[(label, p)]
        metrics = evaluate_m1_m4(result["dbar_i"], dbar_baseline_same_p, D0, result["xj"], p, decile_london)
        m1_m4_rows.append({
            "scenario": label, "alpha": SCENARIOS[label], "p": p,
            **metrics,
            "solver_status": result["status"],
            "solve_seconds": result["solve_seconds"],
        })

m1_m4_table = pd.DataFrame(m1_m4_rows)
print(m1_m4_table.to_string(index=False))

scenario  alpha    p  M1_efficiency_cost_pct  M2_tail_accessibility_m  M3_deprivation_gap_m  M4_equity_allocation_ratio  dbar_baseline_weighted_m solver_status  solve_seconds
       A    0.0 2000                0.000000              1346.582441              5.399892                    0.306168                290.603220       Optimal      11.788757
       B    0.1 2000               -1.462802              1324.591004             12.662058                    0.118292                286.352270       Optimal      15.378544
       C    0.3 2000               -3.717052              1293.683376             25.671555                    0.111334                279.801348       Optimal      13.317843
       D    0.5 2000               -5.169274              1279.391961             36.761497                    0.473168                275.581143       Optimal      14.998408
       A    0.0 5000                0.000000              1346.582441              5.399892                    0.590069      

## 8. K sensitivity (reference scenario only: $\alpha=0.3$, $p=5{,}000$)

$K \in \{0.5K_0,\ K_0,\ 2K_0\}$ -- assesses whether the main allocation pattern is stable
under different assumptions about how much recognised demand one capacity unit can support.

In [14]:
K_sensitivity_rows = []
for K_mult, K_label in [(0.5, "0.5K0"), (1.0, "K0"), (2.0, "2K0")]:
    K_test = K_mult * K0
    result = solve_p_median("D_C", 5_000, K_test, master, coords, cand_default, Uj=150, binary_y=False)
    dist = float(np.average(result["dbar_i"], weights=D0))
    K_sensitivity_rows.append({
        "K_variant": K_label, "K_value": K_test, "status": result["status"],
        "total_slack": float(result["sj"].sum()),
        "D0_weighted_mean_distance_m": dist,
        "n_lsoa_xj_gt_0": int((result["xj"] > 0).sum()),
    })
    print(f"{K_label} (K={K_test:.4f}): status={result['status']}, "
          f"slack={result['sj'].sum():,.1f}, mean_dist={dist:,.1f}m")

K_sensitivity_table = pd.DataFrame(K_sensitivity_rows)
K_sensitivity_table

0.5K0 (K=6.5838): status=Optimal, slack=213,166.9, mean_dist=180.2m
K0 (K=13.1676): status=Optimal, slack=129,178.8, mean_dist=279.8m
2K0 (K=26.3352): status=Optimal, slack=16,297.0, mean_dist=320.4m


,K_variant,K_value,status,total_slack,D0_weighted_mean_distance_m,n_lsoa_xj_gt_0
0,0.5K0,6.583802,Optimal,213166.862829,180.217613,313
1,K0,13.167604,Optimal,129178.818549,279.801348,701
2,2K0,26.335208,Optimal,16297.005079,320.388113,1710


## 9. $U_j$ sensitivity (reference scenario, appendix material)

In [15]:
Uj_sensitivity_rows = []
for Uj_test in [50, 150, 300]:
    result = solve_p_median("D_C", 5_000, K0, master, coords, cand_default, Uj=Uj_test, binary_y=False)
    dist = float(np.average(result["dbar_i"], weights=D0))
    Uj_sensitivity_rows.append({
        "Uj": Uj_test, "status": result["status"],
        "D0_weighted_mean_distance_m": dist,
        "max_xj": int(result["xj"].max()),
        "n_lsoa_xj_gt_0": int((result["xj"] > 0).sum()),
    })
    print(f"Uj={Uj_test}: status={result['status']}, mean_dist={dist:,.1f}m, max xj={result['xj'].max()}")

Uj_sensitivity_table = pd.DataFrame(Uj_sensitivity_rows)
Uj_sensitivity_table

Uj=50: status=Optimal, mean_dist=279.8m, max xj=11
Uj=150: status=Optimal, mean_dist=279.8m, max xj=11
Uj=300: status=Optimal, mean_dist=279.8m, max xj=11


,Uj,status,D0_weighted_mean_distance_m,max_xj,n_lsoa_xj_gt_0
0,50,Optimal,279.801286,11,695
1,150,Optimal,279.801348,11,701
2,300,Optimal,279.801286,11,768


## 10. $k$ (candidate pruning) sensitivity (reference scenario)

Confirms the main allocation pattern is not an artefact of the k-NN neighbourhood size chosen
for computational tractability.

In [16]:
k_sensitivity_rows = []
for k_test in [30, 40, 50]:
    cand_k = build_candidate_sets(coords, k_test) if k_test != K_DEFAULT else cand_default
    result = solve_p_median("D_C", 5_000, K0, master, coords, cand_k, Uj=150, binary_y=False)
    dist = float(np.average(result["dbar_i"], weights=D0))
    k_sensitivity_rows.append({
        "k": k_test, "avg_candidates": np.mean([len(c) for c in cand_k]),
        "status": result["status"], "solve_seconds": result["solve_seconds"],
        "D0_weighted_mean_distance_m": dist,
    })
    print(f"k={k_test}: status={result['status']}, {result['solve_seconds']:.1f}s, mean_dist={dist:,.1f}m")

k_sensitivity_table = pd.DataFrame(k_sensitivity_rows)
k_sensitivity_table

k=30: status=Optimal, 23.3s, mean_dist=217.3m
k=40: status=Optimal, 39.0s, mean_dist=279.8m
k=50: status=Optimal, 52.1s, mean_dist=339.1m


,k,avg_candidates,status,solve_seconds,D0_weighted_mean_distance_m
0,30,30.0,Optimal,23.345881,217.307084
1,40,40.0,Optimal,38.966834,279.801348
2,50,50.0,Optimal,52.133115,339.114514


## 11. Optional service-radius scenario ($r=800$m, reference scenario)

Not part of the 12 core configurations -- a substantive accessibility assumption tested
separately, distinct from the computational $k$-NN pruning above.

In [17]:
result_r800 = solve_p_median("D_C", 5_000, K0, master, coords, cand_default,
                              Uj=150, binary_y=False, service_radius=800.0)
dist_r800 = float(np.average(result_r800["dbar_i"], weights=D0))
dist_no_radius = float(np.average(core_results[("C", 5_000)]["dbar_i"], weights=D0))

print(f"No service radius:  status={core_results[('C', 5_000)]['status']}, mean_dist={dist_no_radius:,.1f}m, "
      f"total_slack={core_results[('C', 5_000)]['sj'].sum():,.1f}")
print(f"r=800m:              status={result_r800['status']}, mean_dist={dist_r800:,.1f}m, "
      f"total_slack={result_r800['sj'].sum():,.1f}")

No service radius:  status=Optimal, mean_dist=279.8m, total_slack=129,178.8
r=800m:              status=Optimal, mean_dist=56.7m, total_slack=151,876.1


## 12. Save outputs

In [18]:
# Wide-format xj results, compatible with 05b_xj_maps.ipynb's expected column naming (xj_<scenario>_p<p>)
results_wide = master[["lsoa_code"]].copy()
for p in P_VALUES:
    for label in SCENARIOS:
        results_wide[f"xj_{label}_p{p}"] = core_results[(label, p)]["xj"]
        results_wide[f"sj_{label}_p{p}"] = core_results[(label, p)]["sj"]

results_wide.to_csv(os.path.join(OUTPUT_DIR, "p_median_results.csv"), index=False)
m1_m4_table.to_csv(os.path.join(OUTPUT_DIR, "m1_m4_results.csv"), index=False)
run_log_df.to_csv(os.path.join(OUTPUT_DIR, "p_median_run_log.csv"), index=False)
K_sensitivity_table.to_csv(os.path.join(OUTPUT_DIR, "K_sensitivity.csv"), index=False)
Uj_sensitivity_table.to_csv(os.path.join(OUTPUT_DIR, "Uj_sensitivity.csv"), index=False)
k_sensitivity_table.to_csv(os.path.join(OUTPUT_DIR, "k_sensitivity.csv"), index=False)

print("Saved:")
for fname in ["p_median_results.csv", "m1_m4_results.csv", "p_median_run_log.csv",
              "K_sensitivity.csv", "Uj_sensitivity.csv", "k_sensitivity.csv"]:
    print(" -", os.path.join(OUTPUT_DIR, fname))

Saved:
 - /Users/alexia/Documents/CASA/Dissertation/05_processed/p_median_results.csv
 - /Users/alexia/Documents/CASA/Dissertation/05_processed/m1_m4_results.csv
 - /Users/alexia/Documents/CASA/Dissertation/05_processed/p_median_run_log.csv
 - /Users/alexia/Documents/CASA/Dissertation/05_processed/K_sensitivity.csv
 - /Users/alexia/Documents/CASA/Dissertation/05_processed/Uj_sensitivity.csv
 - /Users/alexia/Documents/CASA/Dissertation/05_processed/k_sensitivity.csv


## 13. Update pipeline_summary.csv

In [19]:
pipeline_summary_path = os.path.join(OUTPUT_DIR, "pipeline_summary.csv")
pipeline_summary = pd.read_csv(pipeline_summary_path)

new_rows = pd.DataFrame([
    {"output": "p_median_results.csv", "rows": len(results_wide), "observation_unit": "LSOA"},
    {"output": "m1_m4_results.csv", "rows": len(m1_m4_table), "observation_unit": "alpha x p configuration"},
])
for _, row in new_rows.iterrows():
    mask = pipeline_summary["output"] == row["output"]
    if mask.any():
        pipeline_summary.loc[mask, ["rows", "observation_unit"]] = [row["rows"], row["observation_unit"]]
    else:
        pipeline_summary = pd.concat([pipeline_summary, pd.DataFrame([row])], ignore_index=True)

pipeline_summary.to_csv(pipeline_summary_path, index=False)
print(pipeline_summary.to_string(index=False))

                        output   rows        observation_unit
       census_london_clean.csv   4994                    LSOA
          imd_london_clean.csv   4994                    LSOA
       evse_registry_clean.csv  38358                    EVSE
         osev_london_clean.csv  23008       charging location
              zapmap_clean.csv 524116        charging session
      evse_utilisation_all.csv  38358                    EVSE
location_utilisation_clean.csv  23008       charging location
       location_lsoa_clean.csv  23008       charging location
         lsoa_supply_clean.csv   4994                    LSOA
     cleaning_decision_log.csv      7       cleaning decision
             demand_london.csv   4994                    LSOA
          p_median_results.csv   4994                    LSOA
             m1_m4_results.csv     12 alpha x p configuration
